# Day 03 - Schema and Read/Write

**Topic:** Schema and Read/Write  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** อ่านข้อมูลด้วย schema ที่ควบคุมได้ และเขียนผลลัพธ์กลับเป็น file/table ใน Lakehouse

วันนี้เราจะฝึกเรื่องที่เจอบ่อยมากในงาน Data Engineering จริง: การอ่าน raw file เข้ามาเป็น DataFrame, ตรวจ schema, กำหนด schema เองด้วย `StructType`, เปรียบเทียบกับ `inferSchema`, และเขียนผลลัพธ์กลับไปเป็น Parquet หรือ Lakehouse table

เป้าหมายของวันนี้ไม่ใช่แค่ “อ่านไฟล์ได้” แต่ต้องเริ่มมี production mindset ว่า schema คือ contract ระหว่าง source data กับ pipeline ของเรา

## Goal of the day

1. อ่าน CSV ด้วยทั้ง **schema inference** และ **explicit schema**
2. สร้าง schema เองด้วย `StructType` และ `StructField`
3. เปรียบเทียบผลลัพธ์ของ `inferSchema` กับ explicit schema
4. เข้าใจ basic read/write pattern สำหรับ CSV, Parquet และ Lakehouse table
5. เริ่มเห็นว่าทำไม production pipeline ไม่ควรปล่อยให้ Spark เดา schema เองตลอดเวลา

## Why it matters in production

ใน production pipeline เรื่อง schema สำคัญมาก เพราะ:

- Source file อาจเปลี่ยน type หรือ format โดยไม่แจ้งล่วงหน้า
- `inferSchema` อาจเดา type ไม่ตรงกับ business meaning เช่น `customer_id = 001` ถูกอ่านเป็น integer แล้ว leading zero หาย
- Column ที่มีค่าผิด format ปะปน อาจทำให้ Spark infer เป็น `string` ทั้ง column
- Explicit schema ทำให้ pipeline behavior predictable กว่า
- การ write output ต้องเลือก format และ mode ให้เหมาะ เช่น Parquet, Delta table, append, overwrite
- ก่อนเขียน table ควรรู้ schema ชัดเจน เพื่อไม่สร้าง downstream issue

Production takeaway:

> Schema is a data contract. ถ้าเราไม่ควบคุม schema ตั้งแต่ ingestion, ปัญหาจะไหลต่อไปถึง Silver, Gold, report และ downstream consumers

## Key concepts

| Concept | Meaning |
|---|---|
| DataFrame schema | โครงสร้างของ DataFrame เช่น column name, data type, nullable |
| `inferSchema` | ให้ Spark เดา data type จากข้อมูลตัวอย่างในไฟล์ |
| Explicit schema | กำหนด schema เองด้วย `StructType` เพื่อควบคุม type |
| `StructType` | Object ที่เก็บ list ของ fields ทั้งหมดใน schema |
| `StructField` | Field definition ของแต่ละ column เช่น name, type, nullable |
| CSV | Text-based format อ่านง่าย แต่ type ไม่ชัดเจนในตัวเอง |
| Parquet | Columnar file format เหมาะกับ analytics workload |
| Lakehouse table | Table ที่มี metadata และใช้งานผ่าน table name ได้ |
| `read` | Load data เข้า Spark DataFrame |
| `write` | Save DataFrame กลับไปเป็น file หรือ table |

## Concept explanation

ใน PySpark เวลาอ่านข้อมูลจาก file เช่น CSV, JSON หรือ Parquet เข้ามาเป็น DataFrame สิ่งที่ต้องคิดเสมอคือ **schema ของข้อมูลคืออะไร**

สำหรับ CSV โดยเฉพาะ ไฟล์ไม่ได้บอก data type ที่แท้จริงของแต่ละ column ไว้ชัดเจน ทุกอย่างเริ่มจาก text ก่อน Spark จึงมี 2 วิธีหลัก:

1. **Schema inference**  
   ให้ Spark เดา type ให้เรา เช่น string, integer, double, date  
   วิธีนี้สะดวกตอน explore data แต่ไม่ควรพึ่งเป็น default ใน production

2. **Explicit schema**  
   เรากำหนด schema เอง เช่น `customer_id` เป็น string, `signup_date` เป็น date, `total_spend` เป็น double  
   วิธีนี้เขียนเยอะกว่า แต่ควบคุม behavior ได้ดีกว่า

ตัวอย่าง production issue ที่เจอบ่อย:

- `customer_id` เป็น `"001"` แต่ `inferSchema` อ่านเป็น integer `1`
- Column จำนวนเงินมีค่า `"not_available"` ปนอยู่ ทำให้ Spark เดาเป็น string
- Date format ผิด เช่น `"invalid_date"` ทำให้ parse ไม่ได้
- File รอบใหม่เพิ่ม column หรือเปลี่ยน type แล้ว pipeline พัง

ดังนั้นใน ingestion pipeline จริง เรามักใช้ explicit schema เป็น baseline แล้วค่อยจัดการ bad records / schema drift อย่างตั้งใจใน step ถัดไป


## Mermaid diagram: Schema Read/Write Flow

```mermaid
flowchart LR
    A[Raw CSV file] --> B1[Read with inferSchema]
    A --> B2[Read with explicit StructType schema]

    B1 --> C1[Schema may be guessed incorrectly]
    B2 --> C2[Schema is controlled by pipeline]

    C2 --> D[Validate and clean DataFrame]
    D --> E1[Write Parquet to Lakehouse Files]
    D --> E2[Write Delta Lakehouse table]

    E1 --> F1[Read by file path]
    E2 --> F2[Read by table name]
```

ภาพนี้คือ flow หลักของวันนี้: เราจะสร้าง raw CSV เล็ก ๆ ก่อน แล้วอ่านกลับมา 2 แบบเพื่อดูว่า schema ที่ Spark เดาเองกับ schema ที่เรากำหนดเองต่างกันอย่างไร จากนั้นค่อย write output กลับไปใน Lakehouse

## Setup / imports

ใช้ imports พื้นฐานสำหรับ Day 03:

- `pyspark.sql.functions` สำหรับ transformation
- `StructType`, `StructField` และ data types สำหรับ explicit schema
- helper function เล็ก ๆ เพื่อสร้าง mock CSV ลง Lakehouse `Files/` ใน Fabric

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    BooleanType,
    DateType,
)

print("Spark version:", spark.version)

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 3, Finished, Available, Finished, False)

Spark version: 3.5.5.5.4.20260403.6


## Create mock CSV data in Lakehouse Files

เพื่อให้ notebook นี้รันได้เองใน Microsoft Fabric Notebook เราจะสร้าง mock CSV ลงใน Lakehouse `Files/` ก่อน

> ใน repo จะมีไฟล์ตัวอย่างอยู่ที่ `sample_data/customers/day_03_customers_raw.csv` ด้วย แต่ใน Fabric Notebook cell ด้านล่างจะสร้างไฟล์ให้เอง จึงไม่จำเป็นต้อง upload file เองก่อนรัน

In [2]:
customer_csv_content = """customer_id,customer_name,email,signup_date,total_spend,is_active
001,Alice Wong,alice@example.com,2025-01-05,1200.50,true
002,Bob Smith,bob@example.com,2025-01-07,0.00,false
003,Chalida Tan,chalida@example.com,2025-02-10,500.25,true
004,Danai Lee,danai@example.com,invalid_date,890.00,true
005,Eve Park,eve@example.com,2025-03-15,not_available,false
006,Niran Chai,niran@example.com,2025-03-20,320.00,true
"""

customer_csv_path = "Files/day03_schema_read_write/customers_raw.csv"

notebookutils.fs.mkdirs("Files/day03_schema_read_write")
notebookutils.fs.put(customer_csv_path, customer_csv_content, True)

print("Customer CSV path:", customer_csv_path)

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 4, Finished, Available, Finished, False)

Customer CSV path: Files/day03_schema_read_write/customers_raw.csv


## PySpark code examples

### Example 1: Read CSV with `inferSchema`

เริ่มจากวิธีที่ง่ายที่สุด: ให้ Spark เดา schema เอง

จุดที่ต้องสังเกต:

- `customer_id` ถูก infer เป็น `integer` ทำให้ค่าอย่าง `001`, `002` กลายเป็น `1`, `2` และ leading zero หาย
- `total_spend` ถูก infer เป็น `string` เพราะมีค่า `not_available` ปนอยู่ใน column ที่ควรเป็นตัวเลข
- `signup_date` ยังเป็น `string` และมีค่า `invalid_date` ปนอยู่ จึงควร parse/validate เองในขั้นตอน transformation
- `inferSchema` สะดวกสำหรับ explore data เร็ว ๆ แต่ไม่ควรใช้เป็น contract หลักของ production ingestion

In [7]:
df_customers_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customer_csv_path)
)

df_customers_inferred.show(truncate=False)  # Show full column values without truncating text
df_customers_inferred.printSchema()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 13, Finished, Available, Finished, False)

+-----------+-------------+-------------------+------------+-------------+---------+
|customer_id|customer_name|email              |signup_date |total_spend  |is_active|
+-----------+-------------+-------------------+------------+-------------+---------+
|1          |Alice Wong   |alice@example.com  |2025-01-05  |1200.50      |true     |
|2          |Bob Smith    |bob@example.com    |2025-01-07  |0.00         |false    |
|3          |Chalida Tan  |chalida@example.com|2025-02-10  |500.25       |true     |
|4          |Danai Lee    |danai@example.com  |invalid_date|890.00       |true     |
|5          |Eve Park     |eve@example.com    |2025-03-15  |not_available|false    |
|6          |Niran Chai   |niran@example.com  |2025-03-20  |320.00       |true     |
+-----------+-------------+-------------------+------------+-------------+---------+

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: 

### Example 2: Define explicit schema with `StructType`

ต่อไปเราจะกำหนด schema เอง

สิ่งที่ตั้งใจออกแบบ:

- `customer_id` เป็น `StringType` เพื่อรักษา leading zero เช่น `"001"`
- `signup_date` เป็น `DateType`
- `total_spend` เป็น `DoubleType`
- `is_active` เป็น `BooleanType`

ถ้าค่าบางแถว parse ไม่ได้ เช่น `invalid_date` หรือ `not_available` Spark จะทำให้เป็น `null` ตาม default permissive behavior ซึ่งช่วยให้เราเห็น data issue ชัดขึ้น

In [8]:
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("total_spend", DoubleType(), True),
    StructField("is_active", BooleanType(), True),
])

df_customers_explicit = (
    spark.read
    .option("header", True)
    .schema(customer_schema)
    .csv(customer_csv_path)
)

df_customers_explicit.show(truncate=False)
df_customers_explicit.printSchema()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 14, Finished, Available, Finished, False)

+-----------+-------------+-------------------+-----------+-----------+---------+
|customer_id|customer_name|email              |signup_date|total_spend|is_active|
+-----------+-------------+-------------------+-----------+-----------+---------+
|001        |Alice Wong   |alice@example.com  |2025-01-05 |1200.5     |true     |
|002        |Bob Smith    |bob@example.com    |2025-01-07 |0.0        |false    |
|003        |Chalida Tan  |chalida@example.com|2025-02-10 |500.25     |true     |
|004        |Danai Lee    |danai@example.com  |NULL       |890.0      |true     |
|005        |Eve Park     |eve@example.com    |2025-03-15 |NULL       |false    |
|006        |Niran Chai   |niran@example.com  |2025-03-20 |320.0      |true     |
+-----------+-------------+-------------------+-----------+-----------+---------+

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- tot

### Example 3: Compare inferred schema vs explicit schema

ในงานจริง การ compare schema ช่วยให้เห็นว่า Spark เดา type ตรงกับ expectation หรือไม่

สำหรับ Day 03 ให้ดู 2 จุดหลัก:

- `customer_id` ควรเป็น string เพราะเป็น identifier ไม่ใช่ตัวเลขสำหรับคำนวณ
- `total_spend` ควรเป็น double เพื่อใช้คำนวณต่อได้ แต่ invalid value ควรถูก expose ออกมาเป็น null เพื่อจัดการต่อ

In [9]:
print("Inferred schema:")
df_customers_inferred.printSchema()

print("Explicit schema:")
df_customers_explicit.printSchema()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 15, Finished, Available, Finished, False)

Inferred schema:
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- total_spend: string (nullable = true)
 |-- is_active: boolean (nullable = true)

Explicit schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- total_spend: double (nullable = true)
 |-- is_active: boolean (nullable = true)



### Example 4: Add simple validation columns

หลังจากอ่านด้วย explicit schema แล้ว เราสามารถเริ่มทำ validation เบื้องต้นได้

วันนี้ยังไม่ลงลึก Data Quality เต็มรูปแบบ แต่จะเริ่มสร้าง mindset ว่า:

- อ่านข้อมูลแล้วต้องตรวจ type และ invalid value
- column ที่ parse ไม่ได้จะกลายเป็น null
- ก่อน write output ควรมี flag ง่าย ๆ เพื่อบอกว่า record ใช้งานต่อได้หรือไม่

In [10]:
df_customers_checked = (
    df_customers_explicit
    .withColumn("is_valid_signup_date", F.col("signup_date").isNotNull())
    .withColumn("is_valid_total_spend", F.col("total_spend").isNotNull())
    .withColumn(
        "is_valid_record",
        F.col("is_valid_signup_date") & F.col("is_valid_total_spend")  # Both flags must be true
    )
)

df_customers_checked.show(truncate=False)

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 16, Finished, Available, Finished, False)

+-----------+-------------+-------------------+-----------+-----------+---------+--------------------+--------------------+---------------+
|customer_id|customer_name|email              |signup_date|total_spend|is_active|is_valid_signup_date|is_valid_total_spend|is_valid_record|
+-----------+-------------+-------------------+-----------+-----------+---------+--------------------+--------------------+---------------+
|001        |Alice Wong   |alice@example.com  |2025-01-05 |1200.5     |true     |true                |true                |true           |
|002        |Bob Smith    |bob@example.com    |2025-01-07 |0.0        |false    |true                |true                |true           |
|003        |Chalida Tan  |chalida@example.com|2025-02-10 |500.25     |true     |true                |true                |true           |
|004        |Danai Lee    |danai@example.com  |NULL       |890.0      |true     |false               |true                |false          |
|005        |Eve Par

### Example 5: Select clean columns for output

ก่อน write output เราจะเลือก column ที่ต้องการ และเพิ่ม `ingestion_timestamp` เพื่อให้รู้ว่า record ถูก process เมื่อไหร่

นี่เป็น pattern เล็ก ๆ ที่ใช้บ่อยใน Lakehouse pipeline:

- Keep business columns
- Add technical/audit column
- Write result to file/table

In [11]:
df_customers_cleaned = (
    df_customers_checked
    .select(
        "customer_id",
        "customer_name",
        "email",
        "signup_date",
        "total_spend",
        "is_active",
        "is_valid_record",
        F.current_timestamp().alias("ingestion_timestamp"),
    )
)

df_customers_cleaned.show(truncate=False)
df_customers_cleaned.printSchema()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 17, Finished, Available, Finished, False)

+-----------+-------------+-------------------+-----------+-----------+---------+---------------+--------------------------+
|customer_id|customer_name|email              |signup_date|total_spend|is_active|is_valid_record|ingestion_timestamp       |
+-----------+-------------+-------------------+-----------+-----------+---------+---------------+--------------------------+
|001        |Alice Wong   |alice@example.com  |2025-01-05 |1200.5     |true     |true           |2026-05-30 10:02:53.503125|
|002        |Bob Smith    |bob@example.com    |2025-01-07 |0.0        |false    |true           |2026-05-30 10:02:53.503125|
|003        |Chalida Tan  |chalida@example.com|2025-02-10 |500.25     |true     |true           |2026-05-30 10:02:53.503125|
|004        |Danai Lee    |danai@example.com  |NULL       |890.0      |true     |false          |2026-05-30 10:02:53.503125|
|005        |Eve Park     |eve@example.com    |2025-03-15 |NULL       |false    |false          |2026-05-30 10:02:53.503125|


### Example 6: Write output to Parquet

Parquet เป็น columnar format ที่เหมาะกับ analytical workload มากกว่า CSV เพราะข้อมูลถูกจัดเก็บแยกตาม column ทำให้ Spark อ่านเฉพาะ column ที่ต้องใช้ได้ แทนที่จะ scan ทุก column เหมือน text-based CSV

อีกจุดสำคัญคือ Parquet เก็บ schema และ data type ไว้ในไฟล์ด้วย เช่น string, integer, double หรือ date จึงช่วยให้การอ่านข้อมูลซ้ำมีความ predictable มากกว่า และลดปัญหา type เพี้ยนจาก schema inference

ใน Fabric Lakehouse เราสามารถ write Parquet ลง `Files/` แล้ว read กลับด้วย path ได้

In [12]:
customers_parquet_path = "Files/day03_schema_read_write/output/customers_parquet"

(
    df_customers_cleaned
    .write
    .mode("overwrite")
    .parquet(customers_parquet_path)
)

df_customers_from_parquet = spark.read.parquet(customers_parquet_path)

df_customers_from_parquet.show(truncate=False)
df_customers_from_parquet.printSchema()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 18, Finished, Available, Finished, False)

+-----------+-------------+-------------------+-----------+-----------+---------+---------------+--------------------------+
|customer_id|customer_name|email              |signup_date|total_spend|is_active|is_valid_record|ingestion_timestamp       |
+-----------+-------------+-------------------+-----------+-----------+---------+---------------+--------------------------+
|001        |Alice Wong   |alice@example.com  |2025-01-05 |1200.5     |true     |true           |2026-05-30 10:05:29.288179|
|002        |Bob Smith    |bob@example.com    |2025-01-07 |0.0        |false    |true           |2026-05-30 10:05:29.288179|
|003        |Chalida Tan  |chalida@example.com|2025-02-10 |500.25     |true     |true           |2026-05-30 10:05:29.288179|
|004        |Danai Lee    |danai@example.com  |NULL       |890.0      |true     |false          |2026-05-30 10:05:29.288179|
|005        |Eve Park     |eve@example.com    |2025-03-15 |NULL       |false    |false          |2026-05-30 10:05:29.288179|


### Example 7: Write output to Lakehouse table

นอกจากเขียนเป็น file path แล้ว ใน Lakehouse เรามักเขียนเป็น table เพื่อให้ใช้งานผ่าน table name ได้ เช่น `spark.read.table("table_name")`

สำหรับ learning lab วันนี้ เราจะใช้ test table ชื่อ `day03_customers_cleaned`

> ระวัง: `mode("overwrite")` ควรใช้กับ test table เท่านั้น ใน production ต้องคิดก่อนเสมอว่า overwrite จะลบหรือแทนที่ข้อมูลเดิมหรือไม่

In [13]:
customers_table_name = "day03_customers_cleaned"

(
    df_customers_cleaned
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(customers_table_name)
)

df_customers_from_table = spark.read.table(customers_table_name)

df_customers_from_table.show(truncate=False)
df_customers_from_table.printSchema()

print("Table row count:", df_customers_from_table.count())

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 19, Finished, Available, Finished, False)

+-----------+-------------+-------------------+-----------+-----------+---------+---------------+--------------------------+
|customer_id|customer_name|email              |signup_date|total_spend|is_active|is_valid_record|ingestion_timestamp       |
+-----------+-------------+-------------------+-----------+-----------+---------+---------------+--------------------------+
|001        |Alice Wong   |alice@example.com  |2025-01-05 |1200.5     |true     |true           |2026-05-30 10:07:06.170166|
|002        |Bob Smith    |bob@example.com    |2025-01-07 |0.0        |false    |true           |2026-05-30 10:07:06.170166|
|003        |Chalida Tan  |chalida@example.com|2025-02-10 |500.25     |true     |true           |2026-05-30 10:07:06.170166|
|004        |Danai Lee    |danai@example.com  |NULL       |890.0      |true     |false          |2026-05-30 10:07:06.170166|
|005        |Eve Park     |eve@example.com    |2025-03-15 |NULL       |false    |false          |2026-05-30 10:07:06.170166|


### Example 8: Read table back and run a simple check

หลัง write table แล้วควร read กลับมาตรวจอย่างน้อย:

- row count
- schema
- sample records
- invalid record count ถ้ามี validation flag

นี่คือ habit สำคัญของ Data Engineer เพราะ write success ไม่ได้แปลว่า data ถูกต้องเสมอ

In [15]:
invalid_customer_count = (
    df_customers_from_table
    .filter(~F.col("is_valid_record"))  # ~ means NOT, so keep records where is_valid_record is false
    .count()
)

print("Invalid customer count:", invalid_customer_count)

df_customers_from_table.groupBy("is_valid_record").count().show()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 21, Finished, Available, Finished, False)

Invalid customer count: 2
+---------------+-----+
|is_valid_record|count|
+---------------+-----+
|           true|    4|
|          false|    2|
+---------------+-----+



## Quick recap

| Question | Answer |
|---|---|
| CSV มี schema ในตัวเองชัดเจนไหม? | ไม่ชัดเจน ทุก column เริ่มจาก text |
| `inferSchema` คืออะไร? | ให้ Spark เดา data type จากข้อมูล |
| Explicit schema คืออะไร? | เรากำหนด data type เองด้วย `StructType` |
| ทำไม `customer_id` มักควรเป็น string? | เพราะเป็น identifier ไม่ใช่ number สำหรับคำนวณ และอาจมี leading zero |
| Parquet ต่างจาก CSV ยังไง? | Parquet เป็น columnar format และมี schema/data type เก็บอยู่ในไฟล์ |
| Lakehouse table ต่างจาก file path ยังไง? | Table มี metadata และอ่านผ่าน table name ได้ |
| ก่อน write output ควรเช็คอะไร? | schema, row count, sample data, invalid records |

## Exercises

### Exercise 1: Read transaction CSV with explicit schema

สร้าง transaction CSV เล็ก ๆ แล้วอ่านด้วย explicit schema

Required columns:

- `transaction_id`
- `customer_id`
- `transaction_date`
- `amount`
- `payment_method`
- `status`

Expected result:

- `customer_id` ต้องยังเป็น string
- `transaction_date` ต้องเป็น date
- `amount` ต้องเป็น double
- invalid date หรือ invalid amount ควรถูก parse เป็น null

In [19]:
transaction_csv_content = """transaction_id,customer_id,transaction_date,amount,payment_method,status
T001,001,2025-04-01,1500.50,credit_card,success
T002,002,2025-04-02,0.00,promptpay,failed
T003,003,invalid_date,450.00,bank_transfer,success
T004,999,2025-04-03,not_available,credit_card,pending
T005,006,2025-04-04,780.25,promptpay,success
"""

transaction_csv_path = "Files/day03_schema_read_write/transactions_raw.csv"

notebookutils.fs.mkdirs("Files/day03_schema_read_write")
notebookutils.fs.put(transaction_csv_path, transaction_csv_content, True)

print("Transaction CSV path:", transaction_csv_path)

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 25, Finished, Available, Finished, False)

Transaction CSV path: Files/day03_schema_read_write/transactions_raw.csv


In [22]:
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("status", StringType(), True),
])

df_transactions = (
    spark.read
    .option("header", True)
    .schema(transaction_schema)
    .csv(transaction_csv_path)
)

df_transactions.show(truncate=False)
df_transactions.printSchema()
print("Transaction row count:", df_transactions.count())

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 28, Finished, Available, Finished, False)

+--------------+-----------+----------------+------+--------------+-------+
|transaction_id|customer_id|transaction_date|amount|payment_method|status |
+--------------+-----------+----------------+------+--------------+-------+
|T001          |001        |2025-04-01      |1500.5|credit_card   |success|
|T002          |002        |2025-04-02      |0.0   |promptpay     |failed |
|T003          |003        |NULL            |450.0 |bank_transfer |success|
|T004          |999        |2025-04-03      |NULL  |credit_card   |pending|
|T005          |006        |2025-04-04      |780.25|promptpay     |success|
+--------------+-----------+----------------+------+--------------+-------+

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)

Transaction row count: 5


### Exercise 2: Add validation columns to transactions

เพิ่ม validation columns:

- `is_valid_transaction_date`
- `is_valid_amount`
- `is_success_transaction`
- `is_valid_transaction_record`

Expected result:

- record ที่ `transaction_date` parse ไม่ได้ ต้องมี `is_valid_transaction_date = false`
- record ที่ `amount` parse ไม่ได้ ต้องมี `is_valid_amount = false`
- valid transaction record ต้องเป็น success และมี date/amount ที่ถูกต้อง

In [23]:
df_transactions_checked = (
    df_transactions
    .withColumn("is_valid_transaction_date", F.col("transaction_date").isNotNull())
    .withColumn("is_valid_amount", F.col("amount").isNotNull() & (F.col("amount") > 0))
    .withColumn("is_success_transaction", F.lower(F.col("status")) == F.lit("success"))
    .withColumn(
        "is_valid_transaction_record",
        F.col("is_valid_transaction_date")
        & F.col("is_valid_amount")
        & F.col("is_success_transaction")
    )
)

df_transactions_checked.show(truncate=False)

df_transactions_checked.groupBy("is_valid_transaction_record").count().show()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 29, Finished, Available, Finished, False)

+--------------+-----------+----------------+------+--------------+-------+-------------------------+---------------+----------------------+---------------------------+
|transaction_id|customer_id|transaction_date|amount|payment_method|status |is_valid_transaction_date|is_valid_amount|is_success_transaction|is_valid_transaction_record|
+--------------+-----------+----------------+------+--------------+-------+-------------------------+---------------+----------------------+---------------------------+
|T001          |001        |2025-04-01      |1500.5|credit_card   |success|true                     |true           |true                  |true                       |
|T002          |002        |2025-04-02      |0.0   |promptpay     |failed |true                     |false          |false                 |false                      |
|T003          |003        |NULL            |450.0 |bank_transfer |success|false                    |true           |true                  |false          

### Exercise 3: Write transactions to a Lakehouse table and read back

เขียน `df_transactions_checked` เป็น Lakehouse table ชื่อ `day03_transactions_checked`

Required checks:

- read table กลับด้วย `spark.read.table`
- แสดง schema
- แสดง row count
- group by `is_valid_transaction_record`

Expected result:

- table อ่านกลับมาได้
- row count เท่ากับ source transaction rows
- เห็นจำนวน valid/invalid transaction records

In [24]:
transactions_table_name = "day03_transactions_checked"

(
    df_transactions_checked
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(transactions_table_name)
)

df_transactions_from_table = spark.read.table(transactions_table_name)

df_transactions_from_table.show(truncate=False)
df_transactions_from_table.printSchema()

print("Transaction table row count:", df_transactions_from_table.count())

df_transactions_from_table.groupBy("is_valid_transaction_record").count().show()

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 30, Finished, Available, Finished, False)

+--------------+-----------+----------------+------+--------------+-------+-------------------------+---------------+----------------------+---------------------------+--------------------------+
|transaction_id|customer_id|transaction_date|amount|payment_method|status |is_valid_transaction_date|is_valid_amount|is_success_transaction|is_valid_transaction_record|ingestion_timestamp       |
+--------------+-----------+----------------+------+--------------+-------+-------------------------+---------------+----------------------+---------------------------+--------------------------+
|T001          |001        |2025-04-01      |1500.5|credit_card   |success|true                     |true           |true                  |true                       |2026-05-30 10:20:46.319564|
|T002          |002        |2025-04-02      |0.0   |promptpay     |failed |true                     |false          |false                 |false                      |2026-05-30 10:20:46.319564|
|T003          |003 

## Common mistakes

1. **ใช้ `inferSchema` เป็น default ใน production**  
   สะดวกตอน explore แต่เสี่ยงเมื่อ source data เปลี่ยนหรือมีค่าแปลก ๆ ปนมา

2. **อ่าน identifier เป็น integer**  
   เช่น `customer_id = "001"` ถ้าอ่านเป็น integer จะกลายเป็น `1` และทำให้ join หรือ report ผิดได้

3. **ไม่เช็ค schema หลัง read**  
   ควรใช้ `.printSchema()` เสมอหลังอ่าน source ใหม่ โดยเฉพาะ CSV/JSON

4. **เข้าใจว่า write success แปลว่า data ถูกต้อง**  
   Write สำเร็จแปลว่า Spark save ได้ แต่ไม่ได้แปลว่า schema, count หรือ business rule ถูกต้อง

5. **ใช้ `overwrite` โดยไม่ระวัง**  
   `mode("overwrite")` เหมาะกับ test table หรือ controlled replacement เท่านั้น ถ้าใช้ผิดใน production อาจลบข้อมูลเดิม

6. **สับสนระหว่าง Lakehouse Files กับ Lakehouse Tables**  
   `Files/...` คือ file path ส่วน table ใช้ผ่าน table name เช่น `spark.read.table("day03_customers_cleaned")`

7. **ไม่จัดการ invalid parse result**  
   ถ้า explicit schema parse date/number ไม่ได้ ค่าอาจกลายเป็น null ต้องมี validation step ต่อ


## Expected Output / Success Criteria

เมื่อจบ Day 03 ควรทำได้:

- อ่าน CSV ด้วย `inferSchema` ได้ และอธิบายข้อจำกัดของมันได้
- สร้าง explicit schema ด้วย `StructType` และ `StructField` ได้
- อธิบายได้ว่าทำไม identifier เช่น `customer_id` มักควรเป็น string
- ใช้ `.printSchema()` เพื่อ validate schema หลัง read ได้
- เขียน DataFrame เป็น Parquet ลง Lakehouse `Files/` ได้
- เขียน DataFrame เป็น Lakehouse table ด้วย `.saveAsTable()` ได้
- อ่าน table กลับมาด้วย `spark.read.table()` ได้
- เพิ่ม validation flag เบื้องต้นเพื่อจับ invalid date / invalid amount ได้
- เข้าใจว่า schema เป็น data contract ระหว่าง source, pipeline และ downstream consumer

## Final takeaway

Schema and Read/Write คือพื้นฐานที่ดูง่าย แต่สำคัญมากใน production Lakehouse pipeline

> ถ้า schema ผิดตั้งแต่ตอนอ่าน raw data, transformation ต่อไปจะเริ่มผิดแบบเงียบ ๆ ได้

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- ใช้ `inferSchema` เพื่อ explore ได้ แต่อย่าไว้ใจเป็น production contract
- ใช้ explicit schema เมื่อ source มีความสำคัญหรือใช้ซ้ำใน pipeline
- `customer_id`, `transaction_id`, code ต่าง ๆ มักเป็น string ไม่ใช่ number
- หลัง read ต้องเช็ค schema และ sample records
- หลัง write ต้อง read กลับมาตรวจ row count/schema/result
- เริ่มแยกให้ออกว่า `file path` เหมาะกับ raw/intermediate files ส่วน `Lakehouse table` เหมาะกับ curated data ที่ต้อง query ซ้ำ ใช้งานต่อ และมี metadata ชัดเจน

## Optional cleanup

In [25]:
# spark.sql("DROP TABLE IF EXISTS day03_customers_cleaned")
# spark.sql("DROP TABLE IF EXISTS day03_transactions_checked")

# Optional: remove files from Lakehouse Files if needed
# notebookutils.fs.rm("Files/day03_schema_read_write", True)
# mssparkutils.fs.rm("Files/day03_schema_read_write", True)

StatementMeta(, a8400feb-b30b-4153-9f4a-c997b2d77e19, 31, Finished, Available, Finished, False)